# Загрузка Датасета

In [1]:
import torchvision
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])

In [2]:
data_path = '/kaggle/input/datasets/pankrzysiu/cifar10-python' 

train_set = torchvision.datasets.CIFAR10(root=data_path, train=True, download=False, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(root=data_path, train=False, download=False, transform=transform_test)

## Создание загрузчиков

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

# Загрузка модели

In [4]:
import timm

In [5]:
MODEL_NAME = 'vit_tiny_patch16_224'

In [6]:
model = model = timm.create_model(
        'vit_tiny_patch16_224', 
        pretrained=False, 
        num_classes=10, 
        in_chans=3,
        img_size=32, 
        patch_size=4
    )

# Обучение

In [7]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [8]:
epochs = 100
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_58/4221026933.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
for epoch in range(epochs):
    print(f"\nЭпоха {epoch+1}/{epochs}")
    print(f"Текущий Learning Rate: {scheduler.get_last_lr()[0]:.6f}")

    # --- ТРЕНИРОВКА ---
    model.train()
    train_loss, train_correct = 0.0, 0

    for images, labels in tqdm(train_loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Контекст-менеджер смешанной точности
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Обертка обратного прохода и шага оптимизатора через scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()

    # --- ВАЛИДАЦИЯ ---
    model.eval()
    val_loss, val_correct = 0.0, 0

    # Градиенты на валидации не нужны, отключаем для экономии памяти
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Eval", leave=False):
            images, labels = images.to(device), labels.to(device)

            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()

    # Шаг шедулера делаем в конце эпохи
    scheduler.step()

    # Вывод метрик
    train_acc = train_correct / len(train_loader.dataset)
    val_acc = val_correct / len(test_loader.dataset)

    print(f"Train Loss: {train_loss / len(train_loader.dataset):.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss / len(test_loader.dataset):.4f} | Val Acc: {val_acc:.4f}")


Эпоха 1/100
Текущий Learning Rate: 0.001000


Train:   0%|          | 0/391 [00:00<?, ?it/s]

/tmp/ipykernel_58/259934497.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Eval:   0%|          | 0/79 [00:00<?, ?it/s]

/tmp/ipykernel_58/259934497.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

Eval:   0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0067 | Train Acc: 0.9984
Val Loss: 1.1006 | Val Acc: 0.8086

Эпоха 100/100
Текущий Learning Rate: 0.000000


Train:   0%|          | 0/391 [00:00<?, ?it/s]

Eval:   0%|          | 0/79 [00:00<?, ?it/s]

Train Loss: 0.0071 | Train Acc: 0.9980
Val Loss: 1.1009 | Val Acc: 0.8087


In [ ]:
torch.save(model.cpu().state_dict(), 'vit_t_cifar10.pth')